In [6]:
from sqlalchemy import create_engine,text
import pandas as pd

In [3]:
host = "127.0.0.1"
password = "dbms%40123"
engine = create_engine(
    f"mysql+pymysql://root:{password}@{host}:3306/Movie_Analytis"
    #mysql+pymysql://<username>:<password>@<host>/<dbname>[?<options>]
)

connection = engine.connect()

In [5]:
q1=text("""select M.Title,M.Release_Date FROM MOVIES M INNER JOIN Movie_Genres MG ON M.Movie_ID = MG.Movie_ID  WHERE MG.Genre_ID = (SELECT Genre_ID FROM Genres WHERE UCASE(Genres_NAME) LIKE '%ACTION%') and YEAR(M.Release_Date) > 2015;""")

q2=text("""SELECT M.Title, GROUP_CONCAT(G.Genres_name ORDER BY G.Genres_Name SEPARATOR ' ') AS Genres
FROM Movies M
INNER JOIN Movie_Genres MG ON M.Movie_ID = MG.Movie_ID
INNER JOIN Genres G ON MG.Genre_ID = G.Genre_ID
GROUP BY M.Movie_ID, M.Title
ORDER BY M.Revenue DESC LIMIT 10;""")

q3=text("""SELECT G.Genres_Name,AVG(Budget),AVG(Revenue) FROM MOVIES M INNER JOIN MOVIE_Genres MG ON M.Movie_ID = MG.Movie_ID INNER JOIN Genres G ON MG.Genre_ID = G.Genre_ID group by G.Genres_Name ORDER BY AVG(M.Revenue) DESC;""")

q4=text("""SELECT C.Actor_Name,COUNT(DISTINCT M.Movie_ID) As Appearance FROM Movies M INNER JOIN Cast C ON M.Movie_ID = C.Movie_ID GROUP BY C.Actor_Name ORDER BY COUNT(DISTINCT M.Movie_ID) desc limit 10;""")

q5=text("""SELECT C.Person_Name, Count(DISTINCT M.Movie_ID) As Movie_Count FROM Movies M INNER JOIN Crew C ON M.Movie_ID = C.Movie_ID WHERE ucase(C.Job) = "DIRECTOR" GROUP BY C.Person_Name HAVING COUNT(DISTINCT M.Movie_ID)>3 ORDER BY Movie_Count desc limit 10;""")

q6=text("""SELECT Count(M.Movie_ID),MK.Keyword_Name FROM Movies M INNER JOIN Movie_Keywords MK ON M.Movie_ID = MK.Movie_ID GROUP BY MK.Keyword_Name ORDER BY Count(M.Movie_ID) desc limit 10;""")

q7=text("""SELECT Title FROM Movies where Budget > (SELECT AVG(Budget) FROM Movies) AND Revenue < (SELECT AVG(Revenue) FROM Movies);""")

q8=text("""SELECT DISTINCT Ca.Actor_Name,M.Title from Movies M INNER JOIN CREW Cr ON M.Movie_ID = Cr.Movie_ID INNER JOIN Cast Ca ON M.Movie_ID=Ca.Movie_ID where UCASE(Cr.Person_Name) = UCASE("Christopher Nolan") AND UCASE(Cr.Job) = UCASE("Director");""")

q9=text("""SELECT G.Genres_Name FROM Movies M INNER JOIN Movie_Genres MG ON M.Movie_ID = MG.Movie_ID INNER JOIN Genres G ON MG.Genre_ID = G.Genre_ID GROUP BY G.Genres_Name HAVING COUNT(M.Movie_ID)>=20 ORDER BY AVG(M.Vote_Average) desc LIMIT 1;""")

q10=text("""SELECT M.Title,Count(MK.Keyword_Name) FROM Movies M INNER JOIN Movie_Keywords MK ON M.Movie_ID = MK.Movie_ID GROUP BY M.Movie_ID,M.Title ORDER BY Count(MK.Keyword_Name) desc limit 10;""")

queries_List = [q1,q2,q3,q4,q5,q6,q7,q8,q9,q10]

In [8]:
dfs_result = []

In [9]:
with engine.connect() as connection:
    for x in queries_List:
        df = pd.read_sql(x,connection)
        dfs_result.append(df)